# 05 — Branch-4 results walkthrough

A guided tour of the final Branch-4 outputs. We don't retrain anything here —
we load the artefacts that `scripts/score_candidates.py`, `scripts/discovery_shortlist.py`,
and `scripts/render_vetting.py` produced and walk through what the model found.

What this notebook shows:
1. How many of the 6,200 unconfirmed candidates we managed to score.
2. The probability distribution across the discovery pool.
3. The top-10 picks by `prob_mean` (TESS-side dominated by long-period candidates).
4. The headline vetting figure rendered inline.
5. The "since-confirmed" recall sanity check — 120 candidates that became
   confirmed exoplanets between the training-catalogue build and the snapshot
   date, and how the model would have ranked them.

Full methodology and the comparison-with-published-baselines is in
[`docs/research_report_draft.md`](../docs/research_report_draft.md).

## 1. Load the scored discovery pool

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
scored = pd.read_parquet(ROOT / 'results/candidates_scored.parquet')

print(f'Total candidates: {len(scored)}')
print()
print('By mission × status:')
print(scored.groupby(['mission', 'status']).size().to_string())
print()
ok = scored[scored.status == 'ok']
print(f'Successfully scored: {len(ok)} / {len(scored)} = {100*len(ok)/len(scored):.1f}%')

## 2. Probability distribution

The model's `prob_mean` across the 5,388 successfully scored candidates. After
temperature scaling (T* = 1.275), the right tail is compressed — this is the
calibration step working as intended, not a model failure (see report
'Internal sanity check' section).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear-scale histogram
axes[0].hist(ok.prob_mean, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='τ = 0.5')
axes[0].axvline(0.95, color='green', linestyle='--', alpha=0.5, label='τ = 0.95')
axes[0].set_xlabel('prob_mean')
axes[0].set_ylabel('count')
axes[0].set_title('Probability distribution (linear)')
axes[0].legend()

# By mission
for m, colour in [('TESS', 'C0'), ('Kepler', 'C1')]:
    sub = ok[ok.mission == m]
    axes[1].hist(sub.prob_mean, bins=50, alpha=0.6, label=f'{m} (n={len(sub)})', color=colour)
axes[1].set_xlabel('prob_mean')
axes[1].set_title('By mission')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Threshold counts (ok-only):')
for tau in [0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    n = (ok.prob_mean >= tau).sum()
    print(f'  prob_mean >= {tau:.2f}: {n:5d}  ({100*n/len(ok):.1f}%)')

## 3. Top picks

Sorted by raw `prob_mean`. These are unconfirmed candidates that the model
scores as highly likely to be real transits. The top of the list is dominated
by long-period TESS targets, where classical BLS lacks the statistical power
to flag the candidate over the limited TESS baseline.

In [ ]:
top = ok.sort_values('prob_mean', ascending=False).head(10)
cols = ['tic_id', 'toi', 'name', 'mission', 'period', 'prob_mean', 'prob_std', 'fold_disagree']
print(top[cols].to_string(index=False))

## 4. Headline vetting figure — TOI-4328.01

Six-panel vetting view of the top pick: TOI-4328.01 (TIC 77175217), a
P = 703.79 d, ~800 ppm long-period TESS candidate scoring `prob_mean = 0.989`
with `fold_disagree = 0.006` (tightly agreed across the 5-fold ensemble).

Panels:
- **(top-left)** phase-folded global view (full orbit),
- **(top-mid)** phase-folded local view (zoom on the transit),
- **(top-right)** odd / even depth overlay (EB check),
- **(bottom-left)** BLS periodogram with harmonics marked,
- **(bottom-mid)** centroid shift diagram,
- **(bottom-right)** ensemble probability with MC-Dropout band + per-fold dots.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'docs/figures/toi-4328-01_tic_77175217.png')))

## 5. Since-confirmed recall — sanity check

120 candidates were labelled `PC` when the training catalogue was built but
have since been confirmed by other surveys / follow-up programs. They are not
discoveries by this work — but they're a real-world generalisation check on a
population the model never trained on.

Of those 120, what fraction does the model score above the standard decision
thresholds?

In [ ]:
shortlist = pd.read_parquet(ROOT / 'results/discovery_shortlist.parquet')

if 'since_confirmed' in shortlist.columns:
    since = shortlist[shortlist['since_confirmed']]
else:
    # Fallback: enriched parquet may flag this differently
    since = shortlist[shortlist.get('confirmed_after_training', False) == True]

print(f'Since-confirmed planets: {len(since)}')
print()

# Recall at standard thresholds
rows = []
for tau in [0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    above = (since.prob_mean >= tau).sum()
    pct = 100 * above / len(since)
    rows.append({'threshold': tau, 'recovered': f'{above}/{len(since)}', 'recall_pct': f'{pct:.1f}%'})

recall_tbl = pd.DataFrame(rows)
print(recall_tbl.to_string(index=False))
print()
print(f'Mean prob_mean across since-confirmed: {since.prob_mean.mean():.3f}')

## 6. What this means

The 95.8 % recall at threshold 0.5 is a sanity check that the model
generalises to real planets confirmed after training closed. It is **not**
a benchmark claim against published systems — those use random within-distribution
holdouts at different precision operating points, not a temporal holdout of
since-promoted candidates. See the
[methodological note in the research report](../docs/research_report_draft.md#methodological-note-on-cross-study-recall-comparison)
for the careful comparison.

The deliverable of Branch 4 is the **priority list** itself: 146 high-confidence
unconfirmed candidates (140 TESS, 6 Kepler) ranked for follow-up. The headline
metrics that justify trusting this ranking are the Branch-3 5-fold CV
(ROC-AUC = 0.9555, PR-AUC = 0.9586, Brier = 0.0905) reported in the methodology
section of the research report and trained via `scripts/train_model.py
model=cnn_dualview`.

This notebook closes the project narrative arc: notebooks 01–04 walk through
data exploration, preprocessing, the RF baseline, and the CNN training; this
notebook shows what the model does once trained.